## Day 2 - Baseline LightGBM Model

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.metrics import  accuracy_score 
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from sklearn.model_selection import train_test_split
import lightgbm as lgb

In [ ]:
df = pd.read_csv("fusion_pipeline_output_consolidated.csv", parse_dates=["timestamp"])

print("Dataset Shape:", df.shape)
df.head()

In [ ]:
target_column = "Target"
print(df[target_column].value_counts())

In [ ]:
def evaluate_model(y_true, y_pred):

    print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred), 4))
    print("Recall   :", round(recall_score(y_true, y_pred), 4))
    print("F1 Score :", round(f1_score(y_true, y_pred), 4))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

In [ ]:
drop_cols = [target_column, "Failure Type", "timestamp", "Product ID","UDI"]
X = df.drop(columns=drop_cols)
y = df[target_column]
print("Feature Shape:", X.shape)
print("Target Shape :", y.shape)

In [ ]:
X = pd.get_dummies(X, columns=["Type", "shift"], drop_first=True)

print("Columns after encoding:")
print(X.columns.tolist())

In [ ]:
X.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", col) for col in X.columns]
print("Cleaned columns:")
print(X.columns.tolist())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print("Training Size:", X_train.shape)
print("Testing Size :", X_test.shape)

## Baseline model

In [ ]:
baseline_model = lgb.LGBMClassifier(random_state=42, verbose=-1)
baseline_model.fit(X_train, y_train)
print("\nBASELINE MODEL ")

In [ ]:
y_pred_baseline = baseline_model.predict(X_test)
evaluate_model(y_test, y_pred_baseline)

In [ ]:
print("Training Size:", X_train.shape)
print("Testing Size :", X_test.shape)

print("Recall   :", round(recall_score(y_test, y_pred_baseline), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_baseline), 4))

## Class weight balanced

In [ ]:
weighted_model = lgb.LGBMClassifier(random_state=42, verbose=-1, class_weight="balanced")


In [ ]:
weighted_model.fit(X_train, y_train)

In [ ]:
y_pred_weighted = weighted_model.predict(X_test)

In [ ]:
evaluate_model(y_test, y_pred_weighted)

In [ ]:
from sklearn.model_selection import GridSearchCV

## Recall-optimized

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, -1],
    "learning_rate": [0.05, 0.1],
}

grid_recall = GridSearchCV(
    lgb.LGBMClassifier(random_state=42, verbose=-1, class_weight="balanced"),
    param_grid,
    scoring="recall",
    cv=3,
    n_jobs=-1
)
grid_recall.fit(X_train, y_train)

print("Best params (Recall-optimized):", grid_recall.best_params_)

y_pred_recall = grid_recall.best_estimator_.predict(X_test)
evaluate_model(y_test, y_pred_recall)

## F1-Optimized

In [ ]:
grid_f1 = GridSearchCV(
    lgb.LGBMClassifier(random_state=42, verbose=-1, class_weight="balanced"),
    param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1
)
grid_f1.fit(X_train, y_train)

print("Best params (F1-optimized):", grid_f1.best_params_)

y_pred_f1 = grid_f1.best_estimator_.predict(X_test)
evaluate_model(y_test, y_pred_f1)

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Baseline", "Class Weight Balanced", "Recall-Optimized", "F1-Optimized"],
    "Recall": [
        recall_score(y_test, y_pred_baseline),
        recall_score(y_test, y_pred_weighted),
        recall_score(y_test, y_pred_recall),
        recall_score(y_test, y_pred_f1),
    ],
    "Precision": [
        precision_score(y_test, y_pred_baseline),
        precision_score(y_test, y_pred_weighted),
        precision_score(y_test, y_pred_recall),
        precision_score(y_test, y_pred_f1),
    ],
    "F1": [
        f1_score(y_test, y_pred_baseline),
        f1_score(y_test, y_pred_weighted),
        f1_score(y_test, y_pred_recall),
        f1_score(y_test, y_pred_f1),
    ],
})
comparison.round(4)

In [ ]:

print(comparison.round(4))
print("\nSelected model: F1-Optimized")
print("Reasoning: Recall-only tuning caught more failures (0.93 vs 0.84)")
print("but created 149 false alarms vs only 9 — too costly to act on in practice.")
print("F1-optimized model keeps strong Recall (0.84) with far fewer false alarms.")

### Model Comparison Results

The baseline LightGBM model achieved strong overall performance with a Recall of 0.79 and an F1 Score of 0.86. To improve failure detection, different optimization strategies were tested and compared.

### Key Findings

1. **Using `class_weight="balanced"` improved Recall from 0.79 to 0.84.**

   This indicates that assigning greater importance to the minority failure class helped the model identify more actual failures without requiring extensive tuning.

2. **The Recall-Optimized model achieved the highest Recall (0.93).**

   The model successfully detected 63 out of 68 failure cases, making it the best performer in terms of failure detection alone. However, this improvement came at a significant cost.

3. **Recall optimization generated a large number of false alarms.**

   Although Recall increased substantially, Precision dropped to 0.30 and the model produced approximately 149 false positives. This means many healthy machines would be incorrectly flagged for maintenance, increasing operational costs and reducing trust in the system.

4. **The F1-Optimized model provided the best overall balance.**

   The model maintained a strong Recall of 0.84 while achieving a Precision of 0.86 and an F1 Score of 0.85. It generated only 9 false positives, making it significantly more practical for real-world deployment.

5. **Business impact is more important than maximizing a single metric.**

   While the Recall-Optimized model detected more failures, the large number of unnecessary maintenance alerts would be costly in practice. The F1-Optimized model offers a better trade-off between detecting failures and minimizing false alarms.

### Final Model Selection

The **F1-Optimized LightGBM model** was selected as the final model because it provides a balanced combination of Recall, Precision, and F1 Score while maintaining a very low false positive rate.

### Conclusion

This experiment demonstrates that optimizing solely for Recall can be misleading. A model that catches slightly fewer failures but significantly reduces false alarms is often more valuable in real-world predictive maintenance systems. Therefore, the F1-Optimized model was chosen as the most practical and business-friendly solution.


## Failure Prediction Analysis

In [ ]:
results = X_test.copy()
results["actual"] = y_test.values
results["predicted"] = y_pred_f1
results["original_index"] = y_test.index

print("Results shape:", results.shape)
results.head()

In [ ]:
fp = results[(results["actual"] == 0) & (results["predicted"] == 1)]
print("False Positives:", len(fp))
fp

In [ ]:
fn = results[(results["actual"] == 1) & (results["predicted"] == 0)]
print("False Negatives:", len(fn))
fn

In [ ]:
missed_failure_types = df.loc[fn["original_index"], "Failure Type"]
print("Failure types we're missing:")
print(missed_failure_types.value_counts())

In [ ]:
tp = results[(results["actual"] == 1) & (results["predicted"] == 1)]

print("Tool wear — missed failures (avg):", round(df.loc[fn["original_index"], "Tool wear [min]"].mean(), 2))
print("Tool wear — caught failures (avg):", round(df.loc[tp["original_index"], "Tool wear [min]"].mean(), 2))

In [ ]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": grid_f1.best_estimator_.feature_importances_
}).sort_values("importance", ascending=False)
importance.head(10)

In [ ]:

print("False Positives:", len(fp))
print("False Negatives:", len(fn))
print("\nMissed failure types:")
print(missed_failure_types.value_counts())
print("\nTop 3 features driving predictions:")
print(importance.head(3))

### Key Findings from Failure Prediction Analysis

1. **Tool Wear Failure is the primary weakness of the model.** Out of the 11 missed failure cases (False Negatives), 9 belonged to the *Tool Wear Failure* category. This indicates that the model is not struggling with failure prediction in general, but is specifically less effective at detecting this particular type of failure.

2. **Missed failures showed higher tool wear than detected failures.** The average tool wear of missed failures was approximately **200 minutes**, whereas correctly detected failures had an average tool wear of **144 minutes**. This is a counter-intuitive result because higher tool wear would
